Neural networks in Pytorch:
1. define class which describes the layers of the model
2. forward describes how (training-)data gets passed through the layers
3. train model by pass data and compare prediction
4. go with a lot of batches until model is good enough

In [1]:
import os
import torch
from torch import nn                    # neural network module
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# use an accelerator if available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [5]:
# define the class
    # similar to Datasets the network is a subclass of nn.Module
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()              # super-init is used to initialize the parent class for correct functionality
        self.flatten = nn.Flatten()     # tensor to vector
        self.linear_relu_stack = nn.Sequential( # contains several layers to pass in order
            nn.Linear(28*28, 512),      # weighted matrix multiplication with outcome of 512 values
            nn.ReLU(),                  # allows non linearity
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    # forward defines bahavior --> flatten --> go through layers --> return
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits       # logits = raw output of network

# create instance of model
model = NeuralNetwork().to(device)
print(model)

X = torch.rand(1, 28, 28, device=device)    # random Tensor for test of model
logits = model(X)                           # go through network
pred_propab = nn.Softmax(dim=1)(logits)     # raw logits to actual propability
y_pred = pred_propab.argmax(1)              # get index of highest prop
print(f"Predicted class: {y_pred}")
print(model.linear_relu_stack[0].weight.shape)


NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)
Predicted class: tensor([8])
torch.Size([512, 784])


In [12]:
# Model Layers
    # example of a minibatch passing through the network
input_image = torch.rand(3,28,28)       # batch of 3 images
print(input_image.size())

# nn.flatten
flatten = nn.Flatten()
flat_image = flatten(input_image)   # 28 * 28 = 784 --> batch dimensions untouched
print(flat_image.size())

# nn.linear --> linear transormation with stored weights and biases
    # y = xW[T] + b
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

# nn.ReLU --> pos values untouched & neg values = 0 --> elementwise
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

# nn.Sequential --> orderer container to pass data through in defined order
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28 , 28)
logits = seq_modules(input_image)

# nn.Softmax --> convert raw from logits to scaled values representing propab of each class
    # values between 0-1 and sum is 1 (100%)
softmax = nn.Softmax(dim=1)
pred_propab = softmax(logits)
print(pred_propab.sum(dim=1))
print(pred_propab[0].sum())

torch.Size([3, 28, 28])
torch.Size([3, 784])
torch.Size([3, 20])
Before ReLU: tensor([[ 0.3824,  0.2708,  0.3696,  0.0171,  0.4618,  0.3855,  0.1559,  0.0441,
          0.0215,  0.1033,  0.1952, -0.1468,  0.1583,  0.1273,  0.2874, -0.1516,
          0.2008,  0.0613, -0.6289, -0.2031],
        [ 0.4326,  0.1939,  0.3477,  0.2226,  0.7575,  0.1085,  0.5985, -0.1953,
          0.0349,  0.1317,  0.0138,  0.2607, -0.1975, -0.0947,  0.0947,  0.0887,
          0.0318,  0.2453, -0.4922,  0.1357],
        [ 0.4564,  0.2864,  0.4822,  0.2501,  0.6890,  0.1641,  0.3168, -0.3809,
          0.1374,  0.2361,  0.0334, -0.0014,  0.0614,  0.2600,  0.2540,  0.0914,
          0.1012,  0.1635, -0.3860, -0.1825]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.3824, 0.2708, 0.3696, 0.0171, 0.4618, 0.3855, 0.1559, 0.0441, 0.0215,
         0.1033, 0.1952, 0.0000, 0.1583, 0.1273, 0.2874, 0.0000, 0.2008, 0.0613,
         0.0000, 0.0000],
        [0.4326, 0.1939, 0.3477, 0.2226, 0.7575, 0.1085, 0.5985, 0.00

In [13]:
# Model Parameters
    # subclass of nn.Module --> weighted parameters are memorised automatically
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():        # gets weighted values
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]}\n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0288,  0.0320, -0.0353,  ...,  0.0147,  0.0113, -0.0010],
        [-0.0199, -0.0298,  0.0227,  ..., -0.0120, -0.0076,  0.0165]],
       grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([-0.0334,  0.0269], grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0159,  0.0161, -0.0289,  ..., -0.0158, -0.0388,  0.0103],
        [ 0.0057, -0.0097, -0.0306,  ..., -0.0434, -0.0071,  0.0381]],
       grad_fn=<SliceBackward0>)

Layer: linear_relu_stack.2.bias | Size: 